In questo lavoro analizziamo il fenomeno del churn in un’azienda di telecomunicazioni. Il dataset raccoglie informazioni su 7043 clienti e 21 variabili relative a dati demografici, tipologie di servizio sottoscritto, modalità di pagamento e storico contrattuale. L’obiettivo principale è comprendere i fattori che portano un cliente a lasciare il servizio.

In [7]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
#from sklearn.preprocessing import StandardScaler, LabelEncoder

#cwd = Path(os.getcwd())
#csv_file = cwd / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
#df = pd.read_csv(csv_file)
#print("DataFrame shape:", df.shape)
#print(df.head())

Innanzitutto, di seguito viene riportata una panoramica dettagliata di ciascun attributo presente nel dataset

- [customerID] [Nominale]
Identificativo univoco alfanumerico di ciascun cliente.

- [gender][Nominale]
Genere del cliente: “Male” o “Female”.

- [SeniorCitizen][Nominale]
Indicatore binario (0 = non anziano, 1 = anziano) per clienti oltre i 65 anni.

- [Partner][Nominale]
“Yes” o “No”: presenza di partner o coniuge.

- [Dependents][Nominale]
“Yes” o “No”: presenza di familiari a carico.

- [tenure][Rapporto]
Numero di mesi di permanenza del cliente.

- [PhoneService][Nominale]
“Yes” o “No”: sottoscrizione al servizio di telefonia fissa.

- [MultipleLines][Nominale]
“Yes”, “No” o “No phone service”: linee telefoniche aggiuntive.

- [InternetService][Nominale]
Tipo di connessione: “DSL”, “Fiber optic” o “No”.

- [OnlineSecurity][Nominale]
“Yes”, “No” o “No internet service”: opzione di sicurezza online.

- [OnlineBackup][Nominale]
“Yes”, “No” o “No internet service”: servizio di backup online.

- [DeviceProtection][Nominale]
“Yes”, “No” o “No internet service”: protezione del dispositivo.

- [TechSupport][Nominale]
“Yes”, “No” o “No internet service”: assistenza tecnica.

- [StreamingTV][Nominale]
“Yes”, “No” o “No internet service”: streaming TV.

- [StreamingMovies][Nominale]
“Yes”, “No” o “No internet service”: streaming film.

- [Contract][Ordinale]
Tipo di contratto: “Month-to-month”, “One year”, “Two year”.

- [PaperlessBilling][Nominale]
“Yes” o “No”: fatturazione elettronica.

- [PaymentMethod][Nominale]
Metodo di pagamento: “Electronic check”, “Mailed check”, “Bank transfer (automatic)”, “Credit card (automatic)”.

- [MonthlyCharges][Intervallo]
Addebito mensile totale (in dollari USA).

- [TotalCharges][Intervallo]
Totale fatturato fino all’ultima fatturazione (in dollari USA).

- [Churn][Nominale]
Variabile target: “Yes” se il cliente ha abbandonato, “No” altrimenti.


Carichiamo il file principale


In [8]:
file1_path = 'Telecommunications_Industry/CustomerChurn.xlsx'
df1 = pd.read_excel(file1_path)

df1.head()

,LoyaltyID,Customer ID,Senior Citizen,Partner,Dependents,Tenure,Phone Service,Multiple Lines,Internet Service,Online Security,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn
0,318537,7590-VHVEG,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,152148,5575-GNVDE,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,326527,3668-QPYBK,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,845894,7795-CFOCW,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,503388,9237-HQITU,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Carichiamo il secondo file

In [9]:
file2_path = 'Telecommunications_Industry/Telco_customer_churn_demographics.xlsx'
df_dem = pd.read_excel(file2_path).iloc[3:]

df_dem.head()

,Customer ID,Count,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents
3,4598-XLKNJ,1,Female,78,No,Yes,Yes,Yes,1
4,4846-WHAFZ,1,Female,80,No,Yes,Yes,Yes,1
5,4412-YLTKF,1,Female,72,No,Yes,No,Yes,1
6,0390-DCFDQ,1,Female,76,No,Yes,Yes,Yes,2
7,3445-HXXGF,1,Male,66,No,Yes,Yes,No,0


Adesso uniamo i due dataset usando la colonna in comune Customer ID


In [10]:
merged_df = pd.merge(df1, df_dem, on="Customer ID", how="left")

merged_df.head()

,LoyaltyID,Customer ID,Senior Citizen_x,Partner,Dependents_x,Tenure,Phone Service,Multiple Lines,Internet Service,Online Security,...,Total Charges,Churn,Count,Gender,Age,Under 30,Senior Citizen_y,Married,Dependents_y,Number of Dependents
0,318537,7590-VHVEG,No,Yes,No,1,No,No phone service,DSL,No,...,29.85,No,1.0,Female,36.0,No,No,Yes,No,0.0
1,152148,5575-GNVDE,No,No,No,34,Yes,No,DSL,Yes,...,1889.5,No,1.0,Male,46.0,No,No,No,No,0.0
2,326527,3668-QPYBK,No,No,No,2,Yes,No,DSL,Yes,...,108.15,Yes,1.0,Male,37.0,No,No,No,No,0.0
3,845894,7795-CFOCW,No,No,No,45,No,No phone service,DSL,Yes,...,1840.75,No,1.0,Male,53.0,No,No,No,No,0.0
4,503388,9237-HQITU,No,No,No,2,Yes,No,Fiber optic,No,...,151.65,Yes,1.0,Female,19.0,Yes,No,No,Yes,2.0


In [15]:
# Riepilogo dei valori mancanti
missing_data_summary = merged_df.isnull().sum()
missing_data_summary = missing_data_summary[missing_data_summary > 0].sort_values(ascending=False)

missing_data_summary

Count                   3
Gender                  3
Age                     3
Under 30                3
Senior Citizen_y        3
Married                 3
Dependents_y            3
Number of Dependents    3
dtype: int64